# Which known perturbation does my mutant look like?


**What this is.** The perturbation atlas of instruction 71: every within-study contrast of the uniformly reprocessed RNA-seq is a signature (a shrunken log2 fold change against that study's own control), and a query is ranked against all of them by rank correlation.

**The test.** Four perturbations measured by independent studies, curated by reading the contrast labels. Each member is queried with its own study held out entirely; it passes when another lab's experiment of the same perturbation comes first. Chance is the expected best rank of the partners among all candidates. AP2XII-1 knockdown is two deposits of one system (the same lab), so it is the weakest test of independence and is reported as such.

Built 2026-11-07.

In [1]:
import os, pandas as pd
from parascape import perturbation_atlas as P
store = P.load()
print(store['release'], store['lfc'].shape[1], 'signatures over', store['registry'].study.nunique(), 'studies')
missing = {k: [m for m in v if m not in store['registry'].index] for k, v in GROUPS.items()}
print('members not in the store:', {k: v for k, v in missing.items() if v})

expression_traits_2026_11_06 201 signatures over 28 studies
members not in the store: {'BFD1 stabilised by Shield-1': ['expr_lfc_GSE223877_treatment_standard_medium_3um_shield_1_vs_standard_medium_ve_']}


In [2]:
rec = P.recovery(store, GROUPS)
rec.to_csv(os.path.join(OUT, 'recovery.tsv'), sep='\t', index=False)
summary = rec.groupby('perturbation').agg(queries=('query', 'size'), top1=('top1', 'mean'),
    top5=('top5', 'mean'), median_best_rank=('best_partner_rank', 'median'),
    median_chance_rank=('chance_rank', 'median')).round(2)
summary.to_csv(os.path.join(OUT, 'summary.tsv'), sep='\t')
summary

,queries,top1,top5,median_best_rank,median_chance_rank
perturbation,,,,,
AP2XII-1 knockdown,5,0.6,1,1,66.3
"Alkaline stress against normal medium, wild type",5,0,0.8,3,59.3
BFD1 lost under stress,5,0.4,0.6,3,59.3
BFD1 stabilised by Shield-1,2,1,1,1,98.75


In [3]:
rec[['perturbation', 'study', 'best_partner_rank', 'chance_rank', 'candidates', 'best_partner']]

,perturbation,study,best_partner_rank,chance_rank,candidates,best_partner
0,BFD1 stabilised by Shield-1,GSE139537,1,98.5,196,expr_lfc_GSE223877_treatment_standard_medium_3um_shield_1_vs_standard_medium_ve
1,BFD1 stabilised by Shield-1,GSE223877,1,99,197,expr_lfc_GSE139537_condition_shield1_vs_etoh_genotype_delta_bfd1_dd_bfd1_ty
2,BFD1 lost under stress,GSE132238,6,59.3,177,expr_lfc_GSE223819_treatment_50um_iaa_vs_vehicle_pbs_genotype_me49_tir1_200385
3,BFD1 lost under stress,GSE132238,3,59.3,177,expr_lfc_GSE223819_treatment_50um_iaa_vs_vehicle_pbs_genotype_me49_tir1_200385
4,BFD1 lost under stress,GSE132238,1,59.3,177,expr_lfc_GSE223819_treatment_50um_iaa_vs_vehicle_pbs_genotype_me49_tir1_200385
5,BFD1 lost under stress,GSE223819,1,36.4,181,expr_lfc_GSE132238_genotype_delta_bfd1_vs_wt_condition_stress_time_72_hr
6,BFD1 lost under stress,GSE218921,7,38.8,193,expr_lfc_GSE132238_genotype_delta_bfd1_vs_wt_condition_stress_time_72_hr
7,"Alkaline stress against normal medium, wild type",GSE132238,3,59.3,177,expr_lfc_GSE223869_treatment_alkaline_stress_media_ambient_co2_vs_standard_cult
8,"Alkaline stress against normal medium, wild type",GSE132238,2,59.3,177,expr_lfc_GSE223869_treatment_alkaline_stress_media_ambient_co2_vs_standard_cult
9,"Alkaline stress against normal medium, wild type",GSE132238,2,59.3,177,expr_lfc_GSE223869_treatment_alkaline_stress_media_ambient_co2_vs_standard_cult


### An example query, as a user would see it

The BFD1 knockout under stress from GSE218921, its own study held out: the top ten matches.

In [4]:
q = GROUPS['BFD1 lost under stress'][-1]
hits = P.query(store['lfc'][q], store, exclude_study='GSE218921').head(10)
hits.to_csv(os.path.join(OUT, 'example_query.tsv'), sep='\t', index=False)
hits[['rank', 'study', 'contrast', 'spearman', 'genes']]

,rank,study,contrast,spearman,genes
0,1,GSE223869,"me49delta ku80delta bfd1::bfd1-tydelta bfd2 vs me49delta ku80delta bfd1::bfd1-ty [treatment=alkaline stress media, ambient co2]",0.2805,8133
1,2,GSE223621,pru-ova-tdtomato delta bfd2::hxgprt vs pru-ova-tdtomato,0.2772,8133
2,3,GSE223621,pru-ova-tdtomato delta bfd1::mneongreen vs pru-ova-tdtomato,0.2752,8133
3,4,GSE223819,me49/tir1/208020-mng-maid vs me49/tir1 [treatment=vehicle (pbs)],0.2656,8133
4,5,GSE223877,"me49delta ku80delta bfd1::bfd1-tydelta hxgprt::ptub1-dd-bfd1-tydelta bfd2::tdtomato vs me49delta ku80delta bfd1::bfd1-tydelta hxgprt::ptub1-dd-bfd1-ty [treatment=standard medium, +3um shield-1]",0.239,8133
5,6,GSE223819,me49/tir1/208020-mng-maid vs me49/tir1 [treatment=50um iaa],0.2347,8133
6,7,GSE132238,{delta}bfd1 vs wt [condition=stress; time=72 hr],0.2261,8133
7,8,GSE200962,me49 tgcycp2 bz -aux rep2 vs me49 tgcyc5 bz -aux rep1,0.213,8133
8,9,GSE343877,"treatment: alkaline stress: pH 8.2 medium, 4 days; genotype: ME49, eIF5A-K49R point mutant (non-acetylatable, charge-conservative mimic) | Toxoplasma gondii ME49 bradyzoites, eIF5A K49R mutant, alkaline stress 4 days,",0.2093,8133
9,10,GSE200962,me49 tgcycp2 bz -aux rep2 vs me49 tgcyc5 bz -aux rep2,0.2071,8133
